# 2.3 Linear Algebra

Deep learning models are built almost entirely out of linear-algebraic
operations: dot products, matrix-vector products, matrix-matrix products, and
the **norms** used to measure vectors and errors. This section works through
the basic objects — **scalars**, **vectors**, **matrices**, and their
generalization to **tensors** — and the core operations PyTorch gives us over
them, building toward the products and norms that show up throughout the rest
of the book.

In [1]:
import torch

## 2.3.1 Scalars

A **scalar** is a single numerical value, denoted with a lowercase letter
($x, y, z \in \mathbb{R}$). In PyTorch a scalar is simply a tensor with zero
dimensions (`torch.tensor(3.0)`), and the familiar arithmetic operators —
addition, multiplication, division, exponentiation — all work elementwise on
it exactly as they do on ordinary Python numbers.

In [2]:
# Basic arithmetic operations on scalar tensors (0-dim)
x = torch.tensor(3.0)  # scalar: 0-dim tensor
y = torch.tensor(2.0)  # scalar: 0-dim tensor
x + y, x * y, x / y, x**y  # +, *, /, ** all elementwise on scalars


(tensor(5.), tensor(6.), tensor(1.5000), tensor(9.))

## 2.3.2 Vectors

A **vector** is a fixed-length array of scalars, e.g. $\mathbf{x} \in
\mathbb{R}^n$ for an $n$-dimensional vector — a first-order tensor. We write
vectors in bold lowercase and refer to individual entries by subscript ($x_i$
is the $i$-th element). PyTorch indexes 0-based, so $x_1$ in the math
corresponds to `x[0]` in code; `len(x)` and `x.shape` both report $n$. Worth
keeping straight: **order** is the number of axes (a vector has order 1),
while **dimensionality** is the number of components along an axis — "1-D"
and "1000-dimensional" can both describe the same vector.

In [3]:
# Vector: 1st-order tensor, x in R^4
x = torch.arange(4)       # shape (4,)
x, x[2], len(x), x.shape  # indexing is 0-based; len() and .shape both report dimensionality


(tensor([0, 1, 2, 3]), tensor(2), 4, torch.Size([4]))

## 2.3.3 Matrices

Just as vectors generalize scalars to order one, **matrices** generalize
vectors to order two: $\mathbf{A} \in \mathbb{R}^{m \times n}$ has $m$ rows
and $n$ columns, written in bold uppercase, with individual entries $a_{ij}$
(row $i$, column $j$). When $m = n$ the matrix is **square**. The
**transpose** $\mathbf{A}^\top$ flips rows and columns, so
$(\mathbf{A}^\top)_{ij} = a_{ji}$ — a matrix of shape $(m, n)$ transposes to
shape $(n, m)$.

In [4]:
# Matrix: 2nd-order tensor, A in R^(3x2)
A = torch.arange(6).reshape(3, 2)  # shape (3, 2): 3 rows, 2 columns
A, A.T, A.shape                    # A.T swaps rows/cols -> shape (2, 3)


(tensor([[0, 1],
         [2, 3],
         [4, 5]]),
 tensor([[0, 2, 4],
         [1, 3, 5]]),
 torch.Size([3, 2]))

A square matrix is **symmetric** when it equals its own transpose,
$\mathbf{A} = \mathbf{A}^\top$ (so $a_{ij} = a_{ji}$ for every $i, j$).
Symmetric matrices show up constantly in deep learning — covariance matrices
and Gram matrices are two common examples.

In [5]:
A = torch.tensor([[1, 2, 3], [2, 0, 4], [3, 4, 5]])  # a 3x3 symmetric matrix
A, A == A.T  # symmetric: A equals its own transpose, elementwise


(tensor([[1, 2, 3],
         [2, 0, 4],
         [3, 4, 5]]),
 tensor([[True, True, True],
         [True, True, True],
         [True, True, True]]))

## 2.3.4 Tensors

**Tensors** generalize further to arrays of arbitrary order $n$ — a
vocabulary for objects with more axes than a matrix has. A batch of color
images, for instance, is naturally a 4th-order tensor: (batch, height, width,
channel). We write a general entry of a 3rd-order tensor as $x_{ijk}$. The
example below builds a $(2, 3, 4)$ tensor — think of it as 2 stacked
$3\times4$ matrices.

In [6]:
# int32 is a signed 32-bit integer: range -2^31 to 2^31 - 1 (roughly -2.1B to 2.1B)
torch.arange(24, dtype=torch.int32).reshape(2, 3, 4)  # 3rd-order tensor, shape (2, 3, 4)


tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]], dtype=torch.int32)

## 2.3.5 Basic Properties of Tensor Arithmetic

Elementwise operations on two tensors of the same shape always return a
tensor of that same shape. Addition and subtraction work entry by entry, and
so does elementwise multiplication — the **Hadamard product** $\mathbf{A}
\odot \mathbf{B}$, with $(\mathbf{A} \odot \mathbf{B})_{ij} = a_{ij} b_{ij}$ —
which is *not* the same operation as matrix multiplication (§2.3.10).
`B = A.clone()` allocates fresh memory for `B` rather than aliasing `A`, so
the two tensors can be modified independently.

In [7]:
A = torch.arange(6, dtype=torch.float32).reshape(2, 3)  # redefine A: float32, shape (2, 3), used for the rest of this section
B = A.clone()  # allocate new memory for B, a separate copy of A
A, A + B, A * B, A / B, A**B  # all elementwise; A * B is the Hadamard product A (.) B


(tensor([[0., 1., 2.],
         [3., 4., 5.]]),
 tensor([[ 0.,  2.,  4.],
         [ 6.,  8., 10.]]),
 tensor([[ 0.,  1.,  4.],
         [ 9., 16., 25.]]),
 tensor([[nan, 1., 1.],
         [1., 1., 1.]]),
 tensor([[1.0000e+00, 1.0000e+00, 4.0000e+00],
         [2.7000e+01, 2.5600e+02, 3.1250e+03]]))

To isolate just the Hadamard product on its own:

In [8]:
# A * B is the elementwise (Hadamard) product: (A . B)_ij = a_ij * b_ij
A * B


tensor([[ 0.,  1.,  4.],
        [ 9., 16., 25.]])

Scalars combine with tensors too: `a + X` adds $a$ to every entry, and
`a * X` scales every entry — both are elementwise broadcasts, and neither
changes the tensor's shape.

In [9]:
a = 2
X = torch.arange(24).reshape(2, 3, 4)  # 3rd-order tensor, shape (2, 3, 4)
X, a + X, (a * X).shape  # scalar a broadcasts elementwise; shape is unchanged by a * X


(tensor([[[ 0,  1,  2,  3],
          [ 4,  5,  6,  7],
          [ 8,  9, 10, 11]],
 
         [[12, 13, 14, 15],
          [16, 17, 18, 19],
          [20, 21, 22, 23]]]),
 tensor([[[ 2,  3,  4,  5],
          [ 6,  7,  8,  9],
          [10, 11, 12, 13]],
 
         [[14, 15, 16, 17],
          [18, 19, 20, 21],
          [22, 23, 24, 25]]]),
 torch.Size([2, 3, 4]))

## 2.3.6 Reduction

A **reduction** collapses a tensor along one or more axes by summing (or
averaging) over them. The sum of a vector's entries is $\sum_{i=1}^{n} x_i$;
summing a matrix along `axis=0` collapses the rows (summing down each
column), while `axis=1` collapses the columns (summing across each row).
Reducing over every axis at once (or calling `.sum()` with no `axis`) gives
back the same scalar total either way. The **mean** is just this sum divided
by the number of elements reduced over — `A.mean()` with no axis is
equivalent to `A.sum() / A.numel()`, and passing an `axis` divides by only
that axis's length instead.

In [10]:
x = torch.arange(3, dtype=torch.float32)  # shape (3,)
x, x.sum()  # sum() reduces all elements to a single scalar: sum_i x_i
x.shape


torch.Size([3])

In [11]:
# reduce along one axis at a time: axis=0 sums down columns, axis=1 sums across rows
A.shape, A.sum(axis=0), A.sum(axis=1)  # A is (2, 3) -> axis=0 gives shape (3,), axis=1 gives shape (2,)


(torch.Size([2, 3]), tensor([3., 5., 7.]), tensor([ 3., 12.]))

In [12]:
A.sum(axis=[0, 1]) == A.sum()  # reducing over both axes together is the same as summing everything


tensor(True)

In [13]:
# mean along axis=0 is just the axis-0 sum divided by the number of rows reduced over
A.mean(axis=0), A.sum(axis=0) / A.shape[0]  # both give shape (3,) and match exactly


(tensor([1.5000, 2.5000, 3.5000]), tensor([1.5000, 2.5000, 3.5000]))

## 2.3.7 Non-Reduction Sum

Sometimes we want the sum along an axis *without* collapsing that axis away —
for instance to broadcast-divide each row by its own row sum. Passing
`keepdims=True` keeps the reduced axis in the output with size 1 instead of
dropping it, so the result still broadcasts cleanly against the original
tensor. `cumsum` goes further and performs no reduction at all: it returns
the running (cumulative) sum along an axis, so the output keeps exactly the
input's shape.

In [14]:
row_sums = A.sum(axis=1, keepdims=True)  # axis 1 kept, size 1 -> shape (2, 1)
print('A.shape                           :', A.shape)
print('A.sum(axis=1).shape               :', A.sum(axis=1).shape)  # (2,)   -- axis dropped
print('A.sum(axis=1, keepdims=True).shape:', row_sums.shape)       # (2, 1) -- axis kept
print('A / row_sums (each row now sums to 1):\n', A / row_sums)   # broadcasts (2, 3) / (2, 1)
A.cumsum(axis=0)  # running sum down the rows -- no reduction, shape stays (2, 3)


A.shape                           : torch.Size([2, 3])
A.sum(axis=1).shape               : torch.Size([2])
A.sum(axis=1, keepdims=True).shape: torch.Size([2, 1])
A / row_sums (each row now sums to 1):
 tensor([[0.0000, 0.3333, 0.6667],
        [0.2500, 0.3333, 0.4167]])


tensor([[0., 1., 2.],
        [3., 5., 7.]])

## 2.3.8 Dot Products

The **dot product** (or inner product) of two same-length vectors is
$\mathbf{x}^\top \mathbf{y} = \sum_{i=1}^{d} x_i y_i$ — the sum of their
elementwise products, giving back a single scalar. Dot products underlie
weighted sums (e.g. a weighted average, when $\mathbf{y}$'s entries are
weights that sum to one) and, once vectors are normalized to unit length, the
cosine of the angle between them.

In [15]:
y = torch.ones(3, dtype=torch.float32)  # shape (3,), matches x
x, y, torch.dot(x, y)  # dot product: scalar = sum_i x_i * y_i


(tensor([0., 1., 2.]), tensor([1., 1., 1.]), tensor(3.))

In [16]:
torch.sum(x * y)  # equivalent to torch.dot(x, y): elementwise product, then sum


tensor(3.)

## 2.3.9 Matrix-Vector Products

For $\mathbf{A} \in \mathbb{R}^{m \times n}$ and $\mathbf{x} \in
\mathbb{R}^n$, the **matrix-vector product** $\mathbf{Ax} \in \mathbb{R}^m$
has $i$-th entry $(\mathbf{Ax})_i = \mathbf{a}_i^\top \mathbf{x}$ — the dot
product of $\mathbf{A}$'s $i$-th row with $\mathbf{x}$. Geometrically,
multiplying by $\mathbf{A}$ is a linear map from $\mathbb{R}^n$ to
$\mathbb{R}^m$ — exactly what a fully connected layer computes on a single
input vector. `torch.mv(A, x)` and the `@` operator both require
`A.shape[1] == x.shape[0]` and agree on the result.

In [17]:
# matrix-vector product needs A.shape[1] == x.shape[0]
A, x, A.shape, x.shape, torch.mv(A, x), A@x  # (2, 3) @ (3,) -> shape (2,); torch.mv and @ agree


(tensor([[0., 1., 2.],
         [3., 4., 5.]]),
 tensor([0., 1., 2.]),
 torch.Size([2, 3]),
 torch.Size([3]),
 tensor([ 5., 14.]),
 tensor([ 5., 14.]))

## 2.3.10 Matrix-Matrix Multiplication

Multiplying $\mathbf{A} \in \mathbb{R}^{n \times k}$ by $\mathbf{B} \in
\mathbb{R}^{k \times m}$ produces $\mathbf{C} = \mathbf{AB} \in
\mathbb{R}^{n \times m}$, where each entry is again a dot product,
$c_{ij} = \mathbf{a}_i^\top \mathbf{b}_j$ (row $i$ of $\mathbf{A}$ against
column $j$ of $\mathbf{B}$). This can be seen as $m$ matrix-vector products
stacked side by side, and it is a genuinely different operation from the
elementwise Hadamard product — it needs the *inner* dimensions to match
($k = k$), not the full shapes. The term is often shortened to just "matrix
multiplication," but do not confuse it with the Hadamard product: computing
$\mathbf{AB}$ takes cubic time in the matrix dimensions, versus only
quadratic (one multiply per entry) for a Hadamard product.

In [18]:
B = torch.ones(3, 4)  # shape (3, 4)
torch.mm(A, B), A@B  # (2, 3) @ (3, 4) -> shape (2, 4); torch.mm and @ agree


(tensor([[ 3.,  3.,  3.,  3.],
         [12., 12., 12., 12.]]),
 tensor([[ 3.,  3.,  3.,  3.],
         [12., 12., 12., 12.]]))

## 2.3.11 Norms

A **norm** $\|\mathbf{x}\|$ measures the size of a vector: it is non-negative
($\|\mathbf{x}\| \ge 0$, zero only for the zero vector), rescales linearly
with scalar multiplication ($\|\alpha \mathbf{x}\| = |\alpha| \|\mathbf{x}\|$),
and obeys the triangle inequality
($\|\mathbf{x} + \mathbf{y}\| \le \|\mathbf{x}\| + \|\mathbf{y}\|$). The
family of $\ell_p$ norms,

$$\|\mathbf{x}\|_p = \left(\sum_{i=1}^{n} |x_i|^p\right)^{1/p},$$

covers the two used most often: the **Euclidean** ($\ell_2$) norm at $p=2$,
and the **Manhattan** ($\ell_1$) norm at $p=1$,
$\|\mathbf{x}\|_1 = \sum_{i=1}^{n} |x_i|$ — which weights every entry equally
instead of squaring it, making it less sensitive to outliers than $\ell_2$.
`torch.norm` computes the $\ell_2$ norm by default.

In [19]:
u = torch.tensor([3.0, -4.0])  # a vector in R^2
torch.norm(u)  # L2 norm: sqrt(sum_i u_i^2) = sqrt(3^2 + 4^2) = 5


tensor(5.)

In [20]:
torch.abs(u).sum()  # L1 norm: sum_i |u_i| = |3| + |-4| = 7


tensor(7.)

A quick side-by-side makes the difference concrete: $\ell_1$ sums plain
magnitudes while $\ell_2$ sums squared magnitudes before taking a square
root, so $\ell_2$ is pulled down relative to $\ell_1$ whenever a vector's
mass is spread over more than one entry — the two coincide exactly when only
one entry is nonzero.

In [21]:
vectors = {
    'u = [3, -4]'     : torch.tensor([3.0, -4.0]),
    'v = [1, 1, 1, 1]': torch.tensor([1.0, 1.0, 1.0, 1.0]),
    'w = [5, 0, 0]'   : torch.tensor([5.0, 0.0, 0.0]),
}
for name, vec in vectors.items():
    l1, l2 = torch.abs(vec).sum(), torch.norm(vec)  # L1 = sum|x_i|, L2 = sqrt(sum x_i^2)
    print(f'{name:16s} L1 = {l1.item():.3f}   L2 = {l2.item():.3f}')


u = [3, -4]      L1 = 7.000   L2 = 5.000
v = [1, 1, 1, 1] L1 = 4.000   L2 = 2.000
w = [5, 0, 0]    L1 = 5.000   L2 = 5.000


Norms extend to matrices too, though there they can mean more than one
thing: a matrix is both a bag of entries *and* an operator that stretches
vectors it multiplies, and the ratio by which it can stretch a vector leads
to the (costlier to compute) **spectral norm**. For now we use the
**Frobenius norm**, which treats a matrix as one long vector of its entries
and applies the $\ell_2$ formula,
$\|\mathbf{X}\|_F = \sqrt{\sum_{i=1}^{m}\sum_{j=1}^{n} x_{ij}^2}$ — exactly
what `torch.norm` falls back to when its input is a matrix rather than a
vector.

In [22]:
torch.norm(torch.ones((4, 9)))  # Frobenius norm: sqrt(sum_ij x_ij^2) = sqrt(36) = 6


tensor(6.)

## 2.3.12 Summary

- **Scalars, vectors, matrices, and tensors** are 0th-, 1st-, 2nd-, and
  $n$-th order arrays; PyTorch represents all of them the same way, so shape
  is what distinguishes them.
- **Elementwise arithmetic** (`+`, `*`, `/`, `**`) preserves shape; `*`
  between two same-shaped tensors is the **Hadamard product**, not matrix
  multiplication.
- **Reductions** (`sum`, `mean`) collapse one or more axes: `axis=0` reduces
  down columns, `axis=1` reduces across rows. `keepdims=True` keeps the
  reduced axis as size 1 so the result still broadcasts; `cumsum` performs a
  running sum with no reduction at all.
- The **dot product** $\mathbf{x}^\top\mathbf{y} = \sum_i x_i y_i$ is the
  building block for the **matrix-vector product**
  ($(\mathbf{Ax})_i = \mathbf{a}_i^\top\mathbf{x}$) and the
  **matrix-matrix product** ($c_{ij} = \mathbf{a}_i^\top\mathbf{b}_j$) — each
  just a batch of dot products arranged into a vector or matrix.
- **Norms** measure the magnitude of a vector (or, via the spectral or
  Frobenius norm, a matrix) and are most often applied to the *difference*
  between two vectors to measure how far apart they are. $\ell_2$ is
  Euclidean length, $\ell_1$ sums absolute values, and both are instances of
  the general $\ell_p$ family.
- Watch **shapes and dtypes**: `A.shape[1]` must match `x.shape[0]` for
  `A @ x`; `.clone()` (not plain assignment) is what gives an independent
  copy; and signed integer dtypes like `int32` range over
  $-2^{31}$ to $2^{31}-1$, not $\pm 2^{32}$.